# EyePACS -- Module 1 extra data + Module 2 non-longitudinal pretraining

Uses the resized EyePACS dataset
([kaggle.com/datasets/mohlamin/resized-eyepacs-diabetic-retinopathy-dataset](https://www.kaggle.com/datasets/mohlamin/resized-eyepacs-diabetic-retinopathy-dataset),
`all_labels.csv`, ICDR 0-4) for two things:

* **Part A -- Module 1:** builds an EyePACS grading pickle (patient-level split, both eyes of a
  patient stay in the same split) and, optionally, trains DRG-Net's classifier on it.
* **Part B -- Module 2 stage 1:** runs Module 1's lesion segmenters over (a subset of) EyePACS,
  builds per-stage biomarker profiles (EX/HE/MA/SE area + count, LBS), and pretrains the
  Module 2 Generator/Discriminator on single graded images (StarGAN-style stage translation +
  biomarker-profile loss). Checkpoints go to Drive so **notebook 06** can fine-tune from them
  (`USE_NONLONG_PRETRAIN = True` there).

**Round 4:** the EyePACS Module 1 cache is rebuilt with the fine-tuned segmenters on more images
(`PER_GRADE_SUBSET` 3000, was 1500 -- round 3's biomarker losses only saw 7,500 of 88,700 images), images
are retina-cropped and 256 px, the Generator's loss includes the frozen EyePACS grader
(`LAMBDA_GRADER`), and a stage-control check grades the pretrained Generator's outputs with BOTH Module 1
graders (the Discriminator's own stage head, round 3's 74-100 %, graded real images at 17.5 %).

**Run after notebooks 01-03** (needs the FGADR classifier + segmentation checkpoints on Drive).

**Manual prerequisite:** download the Kaggle zip and upload it to `My Drive/Thesis_Datasets/`
(set `EYEPACS_ZIP` below if you named it differently).

**Round 8** (branch `R8-256px-EyePACSGrader`): a fresh **256 px** pretraining (`ROUND_TAG = '_r8'`, `MODULE2_IMAGE_SIZE = 256`, `CONTINUE_ROUND3_PRETRAIN = False`, no grader loss, 80 epochs, ~12-20 h on an L4, resumes after a disconnect). The EyePACS cache is copied from round 5's folder. The stage-control check uses the EyePACS grader only (the FGADR grader is not usable on new data -- `module1/FGADR_GRADER_DIAGNOSIS.md`). Run this before notebook 06.

**Round 9** (branch `R9-Pretrain-Identity-DiffAug`): first short **test runs** that continue round 8's pretraining for 10 epochs each -- A (identity loss on every step + DiffAugment on the critic), B (A + biomarker loss off), C (control) -- each in its own folder, then a comparison cell that names the **winner** by redness and the no-change test. Then the full pretraining with the winner (`PRETRAIN_MODE = 'full'`, `FULL_RUN`). See the round-9 section below Part B's profiles cell.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Free Drive space: the training notebooks save checkpoints and caches here (several GB in total).
# Deleted or overwritten files stay in the Drive Trash and still count -- empty it if space is low.
import shutil as _shutil
try:
    _du = _shutil.disk_usage('/content/drive/MyDrive')
    print(f'Google Drive: {_du.free / 1e9:.1f} GB free of {_du.total / 1e9:.1f} GB')
    if _du.free < 10e9:
        print('\u26a0 Under 10 GB free on Google Drive -- model files may fail to save. Empty the Trash '
              '(drive.google.com/drive/trash) or free space before training.')
except OSError:
    pass

import getpass, os, glob, json
GITHUB_TOKEN = getpass.getpass('GitHub Personal Access Token (repo scope): ')

REPO_OWNER = 'Mieka068'
REPO_NAME = 'DRProgression'
REPO_BRANCH = 'R9-Pretrain-Identity-DiffAug'  # <-- round 9; 'R8-256px-EyePACSGrader' = round 8, 'R5-Round3Baseline-Fixes' = round 5, 'AllModuleEdits-PreFinals' = rounds 3/4, 'mod-2-NonLongIntegration' = the earlier runs
REPO_CODE_SUBDIR = 'M2-DRProgression-VerM-module1-fgadr-poc'

DRIVE_DATA_DIR = '/content/drive/MyDrive/Thesis_Datasets'
assert os.path.isdir(DRIVE_DATA_DIR), f"Not found: {DRIVE_DATA_DIR}"
MODULE2_RUNS_DIR = os.path.join(DRIVE_DATA_DIR, 'module2_runs')
os.makedirs(MODULE2_RUNS_DIR, exist_ok=True)

# Which Module 1 models the pipeline uses -- set the SAME value in notebooks 04_05 and 06-10:
#   'drgnet' -- notebook 11's DRG-Net replication checkpoints (module1_runs/drgnet_replication/)
#   'poc'    -- notebooks 02/03's original POC checkpoints (module1_runs/)
# Everything built from Module 1's outputs (Module 1 caches, Module 2 runs, figures) gets a
# '_drgnet' suffix for 'drgnet', so the two pipelines never overwrite each other's results.
MODULE1_SOURCE = 'drgnet_combined'
assert MODULE1_SOURCE in ('drgnet_combined', 'drgnet', 'poc'), MODULE1_SOURCE
# New Drive folders for this branch's runs: every Module 1 cache / Module 2 / Module 3 folder below
# gets RUN_SUFFIX, so the mod-2-NonLongIntegration results stay untouched for before/after
# comparison. '' = read/write the earlier folders again.
RUN_SUFFIX = '_prefinals'
RUN_TAG = {'drgnet_combined': '_drgnet_comb', 'drgnet': '_drgnet', 'poc': ''}[MODULE1_SOURCE] + RUN_SUFFIX
# Round 4: every Module 1 cache / Module 2 / Module 3 folder ALSO gets ROUND_TAG, so round 3's results
# (no ROUND_TAG) stay untouched for before/after comparison -- the SAME value in notebooks 04_05 and 06-10.
# Model folders (notebook 11) and the laterality file keep RUN_SUFFIX only. '' = round 3's folders.
# Round 8 (branch R8-256px-EyePACSGrader): a fresh 256 px pretraining in new '_r8' folders (round 5's 128 px run
# can't be continued at 256 px; round 4's 256 px run had the grader loss that painted coloured marks).
# Round 9 (branch R9-Pretrain-Identity-DiffAug): new '_r9' folders. First the short pretraining TEST runs (A/B/C,
# PRETRAIN_MODE = 'tests' below), each continuing round 8's pretraining for 10 epochs in its own folder; then the
# full round-9 pretraining with the winner (PRETRAIN_MODE = 'full'). Round 8's folders are only read.
ROUND_TAG = '_r9'
# Round 5 (branch R5-Round3Baseline-Fixes): round 3's Module 2 settings (128 px, no grader loss, round 3's
# best-epoch rule) + the data fixes (text-safe retina crop, SIFT failures dropped). '_r4' = round 4's folders.
RUN_TAG += ROUND_TAG
# Image preprocessing for every script this notebook runs (fundus_preprocess.py), the same in 04_05-10:
#   DRP_RETINA_CROP '1' = crop to the retina and pad to square before resizing (Tianjin's 3:2 and LongDR's
#     2000x1312 photos were squashed into ovals); '0' = round 3's plain resize.
#   DRP_LONGDR_FOLDER 'FundusImagesColor' = LongDR's raw colour photos. Round 3 used FundusImagesNormalized,
#     on which the lesion segmenters (trained on raw photos) marked 10-30 % of a NORMAL retina as lesion
#     (LBS 0.1-0.34 at grade 0, vs ~0.001 on Tianjin/FIRE) -- 559 of the 917 Module 2 pairs.
os.environ['DRP_RETINA_CROP'] = '1'
os.environ['DRP_LONGDR_FOLDER'] = 'FundusImagesColor'
# Module 2 resolution -- the SAME value in notebooks 06, 08, 09 and 10 (item 4.3; the scripts stop
# with an error on a mismatch).
# Round 4: 'auto' = the size notebook 04_05's grader-resolution check recommends (its
# grader_resolution_check.json in this round's Module 1 cache folder): 256 px when the grader reads real
# Tianjin images clearly better at 256 than at 128 px (kappa +0.05 or more), else 128 (~4x faster). Set a
# number (128 / 256) to override. Run 04_05 to the end BEFORE 09 / 06, so every notebook gets the same size.
MODULE2_IMAGE_SIZE = 256  # round 8: 256 px (notebook 12); round 5: 128 px (round 3); 'auto' = notebook 04_05's recommendation (256 in round 4)
MODULE1_RUNS = os.path.join(DRIVE_DATA_DIR, 'module1_runs', *(['drgnet_replication'] if MODULE1_SOURCE != 'poc' else []))
# Notebook 11 on this branch writes its models to drgnet_replication<RUN_SUFFIX>/ (tiled evaluation,
# 200 segmentation epochs, optional EyePACS grader); use those once they exist.
if MODULE1_SOURCE != 'poc' and os.path.isdir(os.path.join(MODULE1_RUNS + RUN_SUFFIX, 'segmentation')):
    MODULE1_RUNS += RUN_SUFFIX
elif MODULE1_SOURCE != 'poc' and RUN_SUFFIX:
    print(f'\u26a0 {MODULE1_RUNS + RUN_SUFFIX} not found -- using the earlier Module 1 models in {MODULE1_RUNS}. '
          'Run notebook 11 on this branch first for the re-trained ones.')
# Lesion segmenters: <MODULE1_RUNS>/segmentation/<SEG_MODEL_PREFIX><lesion>/model_2.pth.tar --
# FGADR + DDR combined (notebook 11) or FGADR only. The classifier is FGADR's either way.
SEG_MODEL_PREFIX = 'comb_models_' if MODULE1_SOURCE == 'drgnet_combined' else 'models_FGADR_NO_TATL_'
# Round 4: notebook 11's fine-tuned combined segmenters (comb_ft_models_*) once all four exist.
if MODULE1_SOURCE == 'drgnet_combined':
    if all(os.path.isfile(os.path.join(MODULE1_RUNS, 'segmentation', f'comb_ft_models_{_l}', 'model_2.pth.tar'))
           for _l in ('ex', 'he', 'ma', 'se')):
        SEG_MODEL_PREFIX = 'comb_ft_models_'
    elif ROUND_TAG:
        print('\u26a0 Fine-tuned segmenters (comb_ft_models_*) not all on Drive yet -- using the 200-epoch '
              'comb_models_*. Run notebook 11 (RUN["comb_seg_ft"]) first for round 4.')
print('Lesion segmenters:', SEG_MODEL_PREFIX + '<lesion>')
MODULE1_CACHE_DIR = os.path.join(DRIVE_DATA_DIR, 'module1_cache' + RUN_TAG)
os.makedirs(MODULE1_CACHE_DIR, exist_ok=True)
print('Module 1 source:', MODULE1_SOURCE, '| models:', MODULE1_RUNS, '| caches:', MODULE1_CACHE_DIR)
if MODULE2_IMAGE_SIZE == 'auto':
    import json as _json
    _res_json = os.path.join(MODULE1_CACHE_DIR, 'grader_resolution_check.json')
    if os.path.isfile(_res_json):
        _res = _json.load(open(_res_json))
        if _res.get('recommended_image_size'):
            MODULE2_IMAGE_SIZE = int(_res['recommended_image_size'])
        else:  # a check file written before the recommended_image_size field
            _gain = max((_res.get('kappa_gain_high_vs_low') or {0: 0}).values())
            MODULE2_IMAGE_SIZE = max(_res['sizes']) if _gain >= 0.05 else min(_res['sizes'])
        print(f'MODULE2_IMAGE_SIZE = {MODULE2_IMAGE_SIZE} (auto, from {_res_json}: {_res.get("recommendation")})')
    else:
        MODULE2_IMAGE_SIZE = 256
        print(f'\u26a0 MODULE2_IMAGE_SIZE = 256 (auto fallback): {_res_json} not found -- run notebook 04_05 to the end '
              'first, or set MODULE2_IMAGE_SIZE by hand. Every Module 2 notebook must use the same size.')

EYEPACS_ZIP = os.path.join(DRIVE_DATA_DIR, 'resized-eyepacs-diabetic-retinopathy-dataset.zip')  # <-- your zip name
if not os.path.isfile(EYEPACS_ZIP):
    _z = sorted(glob.glob(os.path.join(DRIVE_DATA_DIR, '*eyepacs*.zip')) + glob.glob(os.path.join(DRIVE_DATA_DIR, '*EyePACS*.zip')))
    assert _z, f"No EyePACS zip found in {DRIVE_DATA_DIR} -- upload it there first"
    EYEPACS_ZIP = _z[0]
print('EyePACS zip:', EYEPACS_ZIP)

In [ ]:
# Unzip locally on the Colab VM disk (much faster than reading off Drive). -n = skip existing.
EYEPACS_DIR = '/content/data/eyepacs'
os.makedirs(EYEPACS_DIR, exist_ok=True)
!unzip -q -n "{EYEPACS_ZIP}" -d "{EYEPACS_DIR}"

import pandas as pd
_csv = glob.glob(f'{EYEPACS_DIR}/**/all_labels.csv', recursive=True)
assert _csv, 'all_labels.csv not found in the zip -- check: !ls -R /content/data/eyepacs | head'
LABELS_CSV = _csv[0]
_df = pd.read_csv(LABELS_CSV)
print(LABELS_CSV, _df.shape)
print(_df.head())
print('Columns:', list(_df.columns))  # the loaders auto-detect image/grade columns from these

In [ ]:
# Clone this repo. Skips cleanly if already cloned in this runtime.
%cd /content
if not os.path.isdir(f'/content/{REPO_NAME}'):
    !git clone --branch {REPO_BRANCH} https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git {REPO_NAME}
REPO_CODE_DIR = f'/content/{REPO_NAME}/{REPO_CODE_SUBDIR}'
assert os.path.isdir(REPO_CODE_DIR), f"Not found: {REPO_CODE_DIR} -- check REPO_BRANCH/REPO_CODE_SUBDIR above"
%cd {REPO_CODE_DIR}
!pip install -q pandas openpyxl segmentation-models-pytorch matplotlib scipy torchmetrics torch-fidelity

In [ ]:
# Item 7.3: EyePACS image sizes (the resized Kaggle copy's sizes are undocumented); first 5,000 images.
!python {REPO_CODE_DIR}/tools/dataset_image_sizes.py eyepacs="{EYEPACS_DIR}" --max-images 5000 --out "{MODULE1_CACHE_DIR}/dataset_image_sizes_eyepacs.json"

## Part A -- Module 1: EyePACS grading data

`COMBINE_WITH_FGADR = True` merges FGADR's pickle in (train on both). That changes the test
set, so keep the FGADR-only run from notebook 02 for comparison with the DRG-Net paper.

In [ ]:
COMBINE_WITH_FGADR = False
MAX_PER_GRADE = None   # e.g. 3000 to shorten Module 1 training -- EyePACS is ~73% grade 0

PKL_DIR = os.path.join(DRIVE_DATA_DIR, 'module1_pkl')
os.makedirs(PKL_DIR, exist_ok=True)
# RUN_SUFFIX: built with the fixed Usage handling (blank = original training images -> train;
# the earlier pickle had 0 training images)
EYEPACS_PKL = os.path.join(PKL_DIR, f'eyepacs_classification_pkl{RUN_SUFFIX}.pkl')
_extra = ''
if COMBINE_WITH_FGADR:
    _extra += f' --combine-with "{os.path.join(PKL_DIR, "fgadr_classification_pkl.pkl")}"'
if MAX_PER_GRADE:
    _extra += f' --max-per-grade {MAX_PER_GRADE}'
%cd {REPO_CODE_DIR}
!python module1/dataprep/make_eyepacs_classification_pkl.py --eyepacs-root "{EYEPACS_DIR}" --out "{EYEPACS_PKL}"{_extra}

In [ ]:
# OPTIONAL (long: ~35k training images at 512px x 25 epochs is many hours on Colab -- set
# MAX_PER_GRADE above to shorten; resumes after a disconnect). Trains DRG-Net's classifier on the EyePACS pickle with
# module1/configs/eyepacs_poc.yaml. Same DRG-Net setup + compatibility patches as notebook 02.
# Notebook 11 (RUN['eyepacs_cls'], on by default on this branch) trains the same grader into the same
# folder -- run it in one of the two notebooks. EyePACS is ~73 % grade 0, so sampling is
# progressively_balanced here (DRG-Net's yaml: instance_balanced) unless MAX_PER_GRADE caps train.
RUN_EYEPACS_CLASSIFIER_TRAINING = False
EYEPACS_SAMPLING = 'progressively_balanced'   # None = DRG-Net's instance_balanced

if RUN_EYEPACS_CLASSIFIER_TRAINING:
    %cd /content
    if not os.path.isdir('/content/dr-joint-learning'):
        !git clone https://github.com/DFKI-Interactive-Machine-Learning/dr-joint-learning.git dr-joint-learning
        !cd dr-joint-learning && git checkout 0da1bbe885f438390a0e94ec486283d9b21d5547
    !sed -i 's/fillcolor=aug_args.value_fill/fill=aug_args.value_fill/' /content/dr-joint-learning/dr_classification/data/transforms.py
    !pip install -q munch self-attention-cv pretrainedmodels cnn_finetune ipdb
    # Checkpoint/resume (same patch as notebook 11): the training state is saved every epoch on the
    # VM disk and copied to Drive every 5 epochs and at the end (rewriting it on Drive every epoch
    # would fill the Drive with old copies); re-running this cell after a disconnect continues from
    # the last Drive copy.
    !python {REPO_CODE_DIR}/module1/drgnet_resume_patch.py /content/dr-joint-learning
    import re, shutil
    # Where notebooks 04_05-10 look for CLASSIFIER_RUN = 'eyepacs' (same folder notebook 11 uses).
    _cls_root = (os.path.join(DRIVE_DATA_DIR, 'module1_runs') if MODULE1_SOURCE == 'poc'
                 else os.path.join(DRIVE_DATA_DIR, 'module1_runs', 'drgnet_replication' + RUN_SUFFIX))
    _drive_saves = f'{_cls_root}/classification/eyepacs/saves'
    _local_saves = '/content/drg_local/classification/eyepacs/saves'
    os.makedirs(_drive_saves, exist_ok=True); os.makedirs(_local_saves, exist_ok=True)
    for _f in os.listdir(_drive_saves):  # new runtime: resume from the Drive copy
        if not os.path.isfile(os.path.join(_local_saves, _f)):
            shutil.copy2(os.path.join(_drive_saves, _f), os.path.join(_local_saves, _f))
    _cfg = open(f'{REPO_CODE_DIR}/module1/configs/eyepacs_poc.yaml').read()
    _cfg = re.sub(r"save_path: .*", f"save_path: '{_local_saves}'", _cfg, count=1)
    _cfg = re.sub(r"log_path: .*", "log_path: '/content/drg_local/classification/eyepacs/logs'", _cfg, count=1)
    _cfg = re.sub(r"data_index: .*", f"data_index: '{EYEPACS_PKL}'", _cfg, count=1)
    if EYEPACS_SAMPLING and not MAX_PER_GRADE:
        _cfg = re.sub(r"sampling_strategy: \w+", f"sampling_strategy: {EYEPACS_SAMPLING}", _cfg, count=1)
    open('/content/dr-joint-learning/dr_classification/configs/eyepacs_poc.yaml', 'w').write(_cfg)
    %cd /content/dr-joint-learning/dr_classification
    !DRG_CLS_RESUME=1 DRG_CLS_MIRROR_DIR="{_drive_saves}" DRG_CLS_MIRROR_EVERY=5 python main.py -c configs/eyepacs_poc.yaml -overwrite
    for _f in ('best_validation_weights.pt', 'final_weights.pt', 'resume_state.pt'):
        if os.path.isfile(os.path.join(_local_saves, _f)):
            shutil.copy2(os.path.join(_local_saves, _f), os.path.join(_drive_saves, _f))
    assert os.path.isfile(os.path.join(_drive_saves, 'final_weights.pt')), (
        f'final_weights.pt not saved to {_drive_saves} -- is your Google Drive full? (drive.google.com/settings/storage)')
    %cd {REPO_CODE_DIR}
else:
    print('Skipped (RUN_EYEPACS_CLASSIFIER_TRAINING = False).')

## Part B -- Module 2 stage 1: biomarker profiles + non-longitudinal pretraining

Module 1 over all ~88k images takes hours, so by default a grade-stratified subset
(`PER_GRADE_SUBSET` images per grade) is cached. `--resume` + `--save-every` let you re-run this
cell after a disconnect and continue where it stopped. The classifier's predicted grade isn't
used here (EyePACS has real labels) -- only the lesion masks/stats matter.

In [ ]:
CLASSIFIER_RUN = 'eyepacs'   # Module 1 grader: 'eyepacs' (default, notebook 11) or 'fgadr' -- use the one notebook 04_05's comparison recommends
if not glob.glob(os.path.join(MODULE1_RUNS, 'classification', CLASSIFIER_RUN, 'saves', '*.pt')):
    print(f'\u26a0 No {CLASSIFIER_RUN} grader in {MODULE1_RUNS} yet -- using the FGADR one')
    CLASSIFIER_RUN = 'fgadr'

# Locate the Module 1 classifier + segmentation checkpoints (same discovery logic as
# notebooks 04-08). HE/SE are included automatically once they exist on Drive.
SEG_DIR   = os.path.join(MODULE1_RUNS, 'segmentation')
CLS_SAVES = os.path.join(MODULE1_RUNS, 'classification', CLASSIFIER_RUN, 'saves')
# Classifier input normalisation must match its training config (eyepacs_poc.yaml differs).
CLS_NORM_ARGS = ' --classifier-norm eyepacs' if CLASSIFIER_RUN == 'eyepacs' else ''

# EyePACS: best_validation_weights.pt (picked on the Kaggle public test, kept apart from its test set);
# FGADR: final_weights.pt (its "validation" set IS its test set, so the best-validation pick is optimistic).
_pref  = (['best_validation_weights.pt', 'final_weights.pt'] if CLASSIFIER_RUN == 'eyepacs'
          else ['final_weights.pt', 'best_validation_weights.pt'])
_cands = [os.path.join(CLS_SAVES, n) for n in _pref] + sorted(
    glob.glob(os.path.join(CLS_SAVES, '*.pt')), key=os.path.getmtime, reverse=True)
CLS_CKPT = next((p for p in _cands if os.path.isfile(p)), None)
assert CLS_CKPT, f"No classifier .pt in {CLS_SAVES} -- run notebook 02/03 (MODULE1_SOURCE='poc') or 11 ('drgnet' / 'drgnet_combined') first"

SEG_ARGS = ''
for _lesion in ['EX', 'MA', 'HE', 'SE']:
    _p = os.path.join(SEG_DIR, f'{SEG_MODEL_PREFIX}{_lesion.lower()}', 'model_2.pth.tar')
    if os.path.isfile(_p):
        SEG_ARGS += f' --seg-checkpoint {_lesion}="{_p}"'
    print(f'{_lesion} seg :', _p if os.path.isfile(_p) else 'not trained yet')
assert 'EX=' in SEG_ARGS and 'MA=' in SEG_ARGS, f'EX/MA segmentation checkpoints missing in {SEG_DIR} -- run notebook 03 (poc) or 11 (drgnet) first'
print('classifier:', CLS_CKPT)

In [ ]:
# Images per grade to run Module 1 on; None = all images (hours). Round 4: 3000 (was 1500) -> ~13k images
# (grades 3/4 have ~2,000 each), ~2-3 h on an L4; the biomarker losses only apply to cached images.
PER_GRADE_SUBSET = 3000

from crosssectional_dataset import load_labels, sample_per_grade
_labels = load_labels(LABELS_CSV)
if PER_GRADE_SUBSET:
    _labels = sample_per_grade(_labels, PER_GRADE_SUBSET, seed=0)
IMAGE_LIST = '/content/eyepacs_module1_subset.txt'
_labels['image_id'].to_csv(IMAGE_LIST, index=False, header=False)
print(f'{len(_labels)} images queued, per grade:', _labels['grade'].value_counts().sort_index().tolist())

EYEPACS_CACHE = os.path.join(MODULE1_CACHE_DIR, 'module1_outputs_eyepacs.pt')
# Round 5: reuse round 4's EyePACS cache (the same ~13k images, 2-3 h to rebuild) -- copied, round 4 untouched.
# Its .meta.json (the models / settings it was built with) is copied too: --resume only continues a cache whose
# .meta.json matches, and rebuilds from scratch otherwise. A marker file records that the copy finished, so an
# interrupted run (or one from before this fix, which copied the .pt without its .meta.json and started
# rebuilding) is replaced by a fresh copy; the leftover local copy on the runtime disk is removed as well.
# EyePACS photos have no text overlays, so the crop fix doesn't change it. False = rebuild it.
REUSE_ROUND4_EYEPACS_CACHE = True
# Round 8: copied from round 5's folder (itself round 4's cache: same images, models and 256 px masks), else round 4's.
REUSE_EYEPACS_CACHE_FROM = ('_r8', '_r5', '_r4')  # round 9: round 8's copy first (same images, models, 256 px masks)
_r4_cache = next((EYEPACS_CACHE.replace(ROUND_TAG, _t) for _t in REUSE_EYEPACS_CACHE_FROM
                  if ROUND_TAG not in ('', _t) and os.path.isfile(EYEPACS_CACHE.replace(ROUND_TAG, _t))), None)
_r4_marker = EYEPACS_CACHE + '.copied_from_r4'
if REUSE_ROUND4_EYEPACS_CACHE and _r4_cache and os.path.isfile(_r4_cache) and not os.path.isfile(_r4_marker):
    import shutil
    import sys
    sys.path.insert(0, REPO_CODE_DIR)
    from drive_io import local_twin
    shutil.copyfile(_r4_cache, EYEPACS_CACHE)
    if os.path.isfile(_r4_cache + '.meta.json'):
        shutil.copyfile(_r4_cache + '.meta.json', EYEPACS_CACHE + '.meta.json')
    else:
        print('\u26a0 round 4 cache has no .meta.json -- the script below will rebuild it from scratch')
    if local_twin(EYEPACS_CACHE) != EYEPACS_CACHE and os.path.isfile(local_twin(EYEPACS_CACHE)):
        os.remove(local_twin(EYEPACS_CACHE))  # partial rebuild on the runtime disk -- resume from the copy instead
    assert os.path.getsize(EYEPACS_CACHE) == os.path.getsize(_r4_cache), 'copy incomplete -- is Google Drive full?'
    open(_r4_marker, 'w').write(_r4_cache)
    print(f'✓ Copied the earlier EyePACS cache {_r4_cache} -> {EYEPACS_CACHE}')
%cd {REPO_CODE_DIR}/module1
!python apply_to_progression_data.py --images-dir "{EYEPACS_DIR}" \
    --classifier-checkpoint "{CLS_CKPT}"{SEG_ARGS}{CLS_NORM_ARGS} \
    --image-list "{IMAGE_LIST}" --mask-size {max(256, MODULE2_IMAGE_SIZE)} \
    --out "{EYEPACS_CACHE}" --resume --save-every 200 --quiet
%cd {REPO_CODE_DIR}

In [ ]:
# Which biomarkers characterise each stage / change between adjacent stages.
PROFILES_JSON = os.path.join(MODULE2_RUNS_DIR, f'stage_biomarker_profiles{RUN_TAG}.json')
PROFILES_FIG = os.path.join(MODULE2_RUNS_DIR, f'stage_biomarker_profiles{RUN_TAG}.png')
!python stage_biomarker_profiles.py --module1-cache "{EYEPACS_CACHE}" --labels-csv "{LABELS_CSV}" \
    --out "{PROFILES_JSON}" --fig "{PROFILES_FIG}"

from IPython.display import Image as IPImage, display
display(IPImage(PROFILES_FIG))

## Round 9 -- pretraining test runs (red streaks)

Short test runs that continue round 8's pretraining for 10 epochs with different settings, then a comparison cell that **prints the winner**. Run the notebook once per test run (`TEST_RUN = 'A'`, then `'B'`, optionally `'C'`), then set `PRETRAIN_MODE = 'full'` and `FULL_RUN` to the winner for the full pretraining (the three cells after the settings cell; skipped in test mode).


In [ ]:
# Round 9 -- red streaks. Round 8's pretrained Generator satisfied two easily fooled judges (the critic's stage head: 47.5 %
# on real images but ~100 % "correct" on fakes; the image-only biomarker regressor) with red streaks / texture, and asking
# for stage 0 gave outputs the EyePACS grader read as grade 4 in 40 % of cases. Short TEST runs first (~2.5-3 h each on an
# L4), each continuing round 8's pretraining for TEST_EPOCHS in its own folder (round 8's folder is only read):
#   'A'  identity loss on every image every step + DiffAugment on the critic
#   'B'  A + biomarker loss off (lambda_bio 0) -- is the regressor the source of the red?
#   'C'  control: no changes, same extra epochs -- does extra training alone change anything?
# Run the notebook once per test run (Runtime -> Run all, with TEST_RUN = 'A', then 'B', then optionally 'C'; separate
# sessions or accounts are fine). The comparison cell at the end compares every finished run with round 8 and prints the
# winner. Then set PRETRAIN_MODE = 'full' and FULL_RUN = the winner for the full round-9 pretraining (fresh, 80 epochs).
PRETRAIN_MODE = 'tests'   # 'tests' (now) | 'full' (after the verdict)
TEST_RUN = 'A'            # which test run this session trains: 'A', 'B' or 'C'
FULL_RUN = 'A'            # PRETRAIN_MODE = 'full' only: the winner the comparison cell names
TEST_EPOCHS = 10
TEST_SETTINGS = {
    'A': {'label': 'identity every step + DiffAugment',
          'flags': ' --identity-every-step --diffaugment "color,translation,cutout"'},
    'B': {'label': 'identity every step + DiffAugment + biomarker loss off',
          'flags': ' --identity-every-step --diffaugment "color,translation,cutout" --lambda-bio 0 --lambda-mono 0'},
    'C': {'label': 'control (round 8 settings, same extra epochs)', 'flags': ''},
}
assert PRETRAIN_MODE in ('tests', 'full') and TEST_RUN in TEST_SETTINGS and FULL_RUN in TEST_SETTINGS
R8_NONLONG_DIR = os.path.join(MODULE2_RUNS_DIR, 'nonlong_pretrain' + RUN_TAG.replace(ROUND_TAG, '_r8'))
TESTS_DIR = os.path.join(MODULE2_RUNS_DIR, 'round9_pretrain_tests')  # comparison results (round 8 baseline check, verdict)
os.makedirs(TESTS_DIR, exist_ok=True)
def test_dir(run):
    return os.path.join(MODULE2_RUNS_DIR, f'nonlong_pretrain{RUN_TAG}test_{run}')
CLS_NORM = 'eyepacs' if CLASSIFIER_RUN == 'eyepacs' else 'fgadr'
_EYEPACS_GRADER = os.path.join(MODULE1_RUNS, 'classification', 'eyepacs', 'saves', 'best_validation_weights.pt')
assert os.path.isfile(_EYEPACS_GRADER), f'{_EYEPACS_GRADER} not found -- the stage checks need the EyePACS grader'
def stage_check(generator, out_json):
    """tools/stage_control_check.py on the pretraining's held-out EyePACS images (EyePACS grader only), incl. round 9's
    no-change and redness numbers. Same images for every Generator (same seeds), so the runs compare directly."""
    get_ipython().run_line_magic('cd', REPO_CODE_DIR)
    get_ipython().system(f'python tools/stage_control_check.py --generator "{generator}" --image-size {MODULE2_IMAGE_SIZE} '
                         f'--dataset-dir "{EYEPACS_DIR}" --preset eyepacs_resized --module1-cache "{EYEPACS_CACHE}" '
                         f'--grader eyepacs="{_EYEPACS_GRADER}:eyepacs" --per-grade 40 --out "{out_json}"')
print(f'PRETRAIN_MODE = {PRETRAIN_MODE!r}' + (f' | TEST_RUN = {TEST_RUN!r}: {TEST_SETTINGS[TEST_RUN]["label"]} -> {test_dir(TEST_RUN)}'
      if PRETRAIN_MODE == 'tests' else f' | FULL_RUN = {FULL_RUN!r}: {TEST_SETTINGS[FULL_RUN]["label"]}'))


In [ ]:
if PRETRAIN_MODE == 'full':
    # Stage-1 pretraining. Saved straight to Drive so notebook 06 can fine-tune from it in a later
    # session. NUM_EPOCHS: 40 (was 20); lower NUM_EPOCHS / SAMPLES_PER_EPOCH for a quicker first run.
    # Training state is saved every epoch on the runtime's disk and copied to Drive (resume_state.pt)
    # every 10 epochs and after the last one -- after a disconnect, re-run this cell and it continues
    # from the last Drive copy; raising NUM_EPOCHS later continues the same run. To start over, delete
    # it and restart the runtime.
    # Round 5: 80 -- round 3's pretraining was still improving at epoch 40 (stage-head accuracy on real images
    # 0.175, below chance). With CONTINUE_ROUND3_PRETRAIN it continues round 3's run for 40 more epochs.
    # Round 8: 80 at 256 px (the same budget as round 5's 128 px run, so the size is the only change): roughly
    # 12-20 h on an L4 -- it resumes after a disconnect; lower it for a quicker first look.
    NUM_EPOCHS = 80
    SAMPLES_PER_EPOCH = 20000
    # Round 4: frozen-grader loss with the grader found above (CLS_CKPT -- EyePACS by default; gan_guidance.py).
    # 0 = off (round 3). PRETRAIN_BATCH_SIZE 8 at 256 px with the grader in the loss (16 at 128 px, round 3).
    # At 256 px expect roughly 6-10 h for 40 epochs on an L4 -- it resumes after a disconnect.
    LAMBDA_GRADER = 0.0  # rounds 5, 8 and 9: off, as in round 3
    CLS_NORM = 'eyepacs' if CLASSIFIER_RUN == 'eyepacs' else 'fgadr'
    PRETRAIN_BATCH_SIZE = 16 if MODULE2_IMAGE_SIZE <= 128 else 8
    NONLONG_DIR = os.path.join(MODULE2_RUNS_DIR, 'nonlong_pretrain' + RUN_TAG)
    # Round 9: the full run uses the test winner's settings (FULL_RUN, set in the round-9 settings cell above).
    FULL_FLAGS = TEST_SETTINGS[FULL_RUN]['flags']
    print(f'Full round-9 pretraining with test-run {FULL_RUN} settings: {TEST_SETTINGS[FULL_RUN]["label"]}')
    # Round 5: continue round 3's pretraining (40 epochs, 128 px, no grader loss) instead of starting over. Round 3's
    # resume_state.pt is COPIED into this round's folder first, so round 3's own files are never written to.
    CONTINUE_ROUND3_PRETRAIN = False  # round 8: start fresh at 256 px (round 3's state is 128 px); round 5: True
    ROUND3_NONLONG_DIR = os.path.join(MODULE2_RUNS_DIR, 'nonlong_pretrain' + RUN_TAG[:len(RUN_TAG) - len(ROUND_TAG)])
    _state = os.path.join(NONLONG_DIR, 'resume_state.pt')
    if CONTINUE_ROUND3_PRETRAIN and ROUND_TAG and not os.path.isfile(_state):
        _r3_state = os.path.join(ROUND3_NONLONG_DIR, 'resume_state.pt')
        assert os.path.isfile(_r3_state), f'{_r3_state} not found -- set CONTINUE_ROUND3_PRETRAIN = False to start over'
        assert MODULE2_IMAGE_SIZE == 128, 'round 3 pretrained at 128 px -- set MODULE2_IMAGE_SIZE = 128'
        import shutil
        os.makedirs(NONLONG_DIR, exist_ok=True)
        shutil.copy(_r3_state, _state)
        print(f'✓ Copied round 3 pretraining state {_r3_state} -> {_state} (round 3 untouched)')
    elif os.path.isfile(_state):
        print(f'Continuing this round\'s own pretraining state: {_state}')

    !python train_module2_nonlong_pretrain.py --image-size {MODULE2_IMAGE_SIZE} --dataset-dir "{EYEPACS_DIR}" --preset eyepacs_resized \
        --module1-cache "{EYEPACS_CACHE}" --profiles "{PROFILES_JSON}" \
        --num-epochs {NUM_EPOCHS} --samples-per-epoch {SAMPLES_PER_EPOCH} --num-workers 2 \
        --batch-size {PRETRAIN_BATCH_SIZE} \
        --grader-loss-checkpoint "{CLS_CKPT}" --grader-loss-norm {CLS_NORM} --lambda-grader {LAMBDA_GRADER}{FULL_FLAGS} \
        --save-dir "{NONLONG_DIR}" --sample-dir "{NONLONG_DIR}/samples" \
        --resume-state "{NONLONG_DIR}/resume_state.pt"
    for _f in ('final-G.ckpt', 'final-D.ckpt', 'resume_state.pt'):
        assert os.path.isfile(os.path.join(NONLONG_DIR, _f)) and os.path.getsize(os.path.join(NONLONG_DIR, _f)) > 0, (
            f'{_f} not found in {NONLONG_DIR}: if the training output above ends with an error, training stopped '
            f'early -- fix that and re-run this cell (it resumes); if training finished, your Google Drive may be '
            f'full -- check drive.google.com/settings/storage and empty the Trash.')
    print('Pretrained Generator/Discriminator saved to Drive:', NONLONG_DIR)
else:
    print("Skipped: PRETRAIN_MODE = 'tests' (the round-9 test cells below run instead). Set PRETRAIN_MODE = 'full' after the verdict for the full pretraining.")


In [ ]:
if PRETRAIN_MODE == 'full':
    # Rows: held-out EyePACS images. Columns: input, then the Generator's stage 0..4 versions.
    _samples = sorted(glob.glob(f'{NONLONG_DIR}/samples/epoch_*.jpg'), key=lambda p: int(p.split('_')[-1].split('.')[0]))
    if _samples:
        print(_samples[-1]); display(IPImage(_samples[-1]))
    _res = os.path.join(NONLONG_DIR, 'nonlong_pretrain_results.json')
    print(json.dumps(json.load(open(_res))['metrics'], indent=2) if os.path.isfile(_res) else 'no results yet')
else:
    print("Skipped: PRETRAIN_MODE = 'tests' (the round-9 test cells below run instead). Set PRETRAIN_MODE = 'full' after the verdict for the full pretraining.")


In [ ]:
if PRETRAIN_MODE == 'full':
    # Round 4 (item R4-3.4): stage control of the pretrained Generator, graded by BOTH Module 1 graders on held-out
    # EyePACS images (the pretraining's own val split): is a stage-k request read as stage k? The grader that was
    # NOT in the Generator's loss (FGADR by default) is the honest number; 'real_accuracy_at_this_size' is the
    # same grader on the real images at this size -- the ceiling. Compare with D's own stage head above.
    # Round 8: EyePACS only. FGADR grades every Tianjin eye 3 or 4 (module1/FGADR_GRADER_DIAGNOSIS.md), so it can't
    # check anything; with LAMBDA_GRADER = 0 the EyePACS grader isn't in G's loss, so its number is the honest one.
    _gr = {'eyepacs': ('eyepacs/saves/best_validation_weights.pt', 'eyepacs')}
    _gargs = ''.join(f' --grader {n}="{os.path.join(MODULE1_RUNS, "classification", p)}:{norm}"'
                     for n, (p, norm) in _gr.items() if os.path.isfile(os.path.join(MODULE1_RUNS, 'classification', p)))
    STAGE_CONTROL_JSON = os.path.join(NONLONG_DIR, 'stage_control_check.json')
    %cd {REPO_CODE_DIR}
    !python tools/stage_control_check.py --generator "{NONLONG_DIR}/final-G.ckpt" --image-size {MODULE2_IMAGE_SIZE} \
        --dataset-dir "{EYEPACS_DIR}" --preset eyepacs_resized --module1-cache "{EYEPACS_CACHE}"{_gargs} \
        --per-grade 40 --out "{STAGE_CONTROL_JSON}"
else:
    print("Skipped: PRETRAIN_MODE = 'tests' (the round-9 test cells below run instead). Set PRETRAIN_MODE = 'full' after the verdict for the full pretraining.")


In [ ]:
# Round 9: round 8's pretrained Generator through the same check (~5 min, once; reused after). The numbers every test run
# is compared against. Written to round9_pretrain_tests/ -- round 8's own folder is not touched.
R8_CHECK_JSON = os.path.join(TESTS_DIR, 'stage_check_r8.json')
if PRETRAIN_MODE != 'tests':
    print('Skipped (PRETRAIN_MODE = "full")')
elif os.path.isfile(R8_CHECK_JSON):
    print('✓ Reusing', R8_CHECK_JSON)
else:
    assert os.path.isfile(os.path.join(R8_NONLONG_DIR, 'final-G.ckpt')), f'round 8 pretraining not found in {R8_NONLONG_DIR}'
    stage_check(os.path.join(R8_NONLONG_DIR, 'final-G.ckpt'), R8_CHECK_JSON)


In [ ]:
# Round 9 test run TEST_RUN: round 8's training state (resume_state.pt, 80 epochs) is COPIED into this run's own folder
# and training continues for TEST_EPOCHS more epochs with the run's settings (round 8's folder is never written to).
# Resumes after a disconnect like the full pretraining: re-run the notebook with the same TEST_RUN. Then the stage check
# (with the no-change / redness numbers) runs on the result.
if PRETRAIN_MODE != 'tests':
    print('Skipped (PRETRAIN_MODE = "full")')
else:
    import shutil
    _dir = test_dir(TEST_RUN)
    os.makedirs(_dir, exist_ok=True)
    _state = os.path.join(_dir, 'resume_state.pt')
    _r8_state = os.path.join(R8_NONLONG_DIR, 'resume_state.pt')
    _r8_epochs = len(json.load(open(os.path.join(R8_NONLONG_DIR, 'nonlong_pretrain_results.json')))['loss_history'])
    if not os.path.isfile(_state):
        assert os.path.isfile(_r8_state), f'{_r8_state} not found -- round 8\'s notebook 09 output'
        shutil.copy(_r8_state, _state)
        assert os.path.getsize(_state) == os.path.getsize(_r8_state), 'copy incomplete -- is Google Drive full?'
        print(f'✓ Copied round 8\'s training state ({_r8_epochs} epochs) -> {_state}')
    else:
        print(f'Continuing this test run\'s own state: {_state}')
    _flags = TEST_SETTINGS[TEST_RUN]['flags']
    PRETRAIN_BATCH_SIZE = 16 if MODULE2_IMAGE_SIZE <= 128 else 8  # round 8's
    %cd {REPO_CODE_DIR}
    !python train_module2_nonlong_pretrain.py --image-size {MODULE2_IMAGE_SIZE} --dataset-dir "{EYEPACS_DIR}" --preset eyepacs_resized \
        --module1-cache "{EYEPACS_CACHE}" --profiles "{PROFILES_JSON}" \
        --num-epochs {_r8_epochs + TEST_EPOCHS} --samples-per-epoch 20000 --num-workers 2 \
        --batch-size {PRETRAIN_BATCH_SIZE}{_flags} \
        --save-dir "{_dir}" --sample-dir "{_dir}/samples" --resume-state "{_state}"
    assert os.path.isfile(os.path.join(_dir, 'final-G.ckpt')), (
        f'final-G.ckpt not found in {_dir}: if the output above ends with an error, fix it and re-run (it resumes); '
        'otherwise check that Google Drive is not full.')
    stage_check(os.path.join(_dir, 'final-G.ckpt'), os.path.join(_dir, 'stage_control_check.json'))


In [ ]:
# Round 9 comparison + VERDICT: every finished test run vs round 8, on the same held-out EyePACS images (EyePACS grader).
#   redness      mean red added inside the retina, averaged over the requested stages (the streaks; lower = fewer)
#   no-change    own stage requested: mean change of the image, and how often the EyePACS grade changes (both -> 0)
#   stage ctrl   requested stage read as that stage (chance 20 %); "0 -> graded 4" = stage-0 requests read as grade 4
# A run FIXES the streaks if its redness is at most REDNESS_FIX x round 8's AND its no-change grade-change rate is not
# higher than round 8's. Winner = the fixing run with the lowest redness (A preferred over B within 10 %: fewer changes).
REDNESS_FIX = 0.5
import pandas as pd
from IPython.display import Image as IPImage, display
if PRETRAIN_MODE != 'tests':
    print('Skipped (PRETRAIN_MODE = "full")')
else:
    _runs = {'round 8': os.path.join(TESTS_DIR, 'stage_check_r8.json')}
    _runs.update({k: os.path.join(test_dir(k), 'stage_control_check.json') for k in TEST_SETTINGS})
    _res = {k: json.load(open(p)) for k, p in _runs.items() if os.path.isfile(p)}
    _missing = [k for k in TEST_SETTINGS if k not in _res]
    def _row(name, r):
        g = r['graders']['eyepacs']
        cm = g['confusion_requested_x_graded']
        return {'run': name, 'settings': TEST_SETTINGS[name]['label'] if name in TEST_SETTINGS else 'round 8 pretrained (80 epochs)',
                'redness (mean)': round(r['mean_redness'], 4),
                **{f'redness {k}': round(v, 4) for k, v in r['redness'].items()},
                'no-change: image change': round(r['no_change']['mean_abs_change'], 4),
                'no-change: grade changed': f"{r['no_change']['grade_changed_rate']['eyepacs']:.0%}",
                'stage ctrl (mean)': f"{g['mean_agreement']:.0%}",
                '0 -> graded 4': f"{cm[0][4] / max(sum(cm[0]), 1):.0%}"}
    if 'round 8' not in _res:
        print('⚠ Round 8 baseline check missing -- run the round-8 baseline cell above first.')
    else:
        display(pd.DataFrame([_row(k, r) for k, r in _res.items()]))
        pd.DataFrame([_row(k, r) for k, r in _res.items()]).to_csv(os.path.join(TESTS_DIR, 'round9_pretrain_tests.csv'), index=False)
        r8 = _res['round 8']
        red8, flip8 = r8['mean_redness'], r8['no_change']['grade_changed_rate']['eyepacs']
        def _fixes(k):
            r = _res[k]
            return (r['mean_redness'] <= REDNESS_FIX * red8
                    and r['no_change']['grade_changed_rate']['eyepacs'] <= flip8 + 1e-9)
        print(f"\nRound 8: redness {red8:.4f} | no-change grade changed {flip8:.0%} | "
              f"stage control {r8['graders']['eyepacs']['mean_agreement']:.0%}")
        for k in TEST_SETTINGS:
            if k in _res:
                r = _res[k]
                print(f"  {k} ({TEST_SETTINGS[k]['label']}): redness {r['mean_redness']:.4f} "
                      f"({r['mean_redness'] / red8:.0%} of round 8) | no-change grade changed "
                      f"{r['no_change']['grade_changed_rate']['eyepacs']:.0%} | stage control "
                      f"{r['graders']['eyepacs']['mean_agreement']:.0%} -> {'FIXES the streaks' if _fixes(k) else 'does not fix them'}")
        print('\nVERDICT:')
        fixing = [k for k in ('A', 'B') if k in _res and _fixes(k)]
        if not any(k in _res for k in ('A', 'B')):
            print('  No test run finished yet -- run this notebook with TEST_RUN = "A" (then "B").')
        elif not fixing:
            done = [k for k in ('A', 'B') if k in _res]
            best = min(done, key=lambda k: _res[k]['mean_redness'])
            print(f"  NO WINNER: no finished run ({', '.join(done)}) halves round 8's redness without making the no-change "
                  f"test worse (best: {best}, {_res[best]['mean_redness'] / red8:.0%} of round 8's redness).")
            if 'C' in _res and _fixes('C'):
                print('  Note: the control C (no changes) DOES halve the redness -- extra training alone helps, the changes do not.')
            if _missing and set(_missing) & {'A', 'B'}:
                print(f"  Still to run: {', '.join(k for k in _missing if k in 'AB')} -- run it before deciding.")
            else:
                print('  The loss changes are not enough on their own: report back before the full pretraining (next suspect:'
                      ' the Generator architecture -- its upsampling can produce stripe / grid textures).')
        else:
            win = min(fixing, key=lambda k: _res[k]['mean_redness'])
            if win == 'B' and 'A' in fixing and _res['A']['mean_redness'] <= 1.1 * _res['B']['mean_redness']:
                win = 'A'  # within 10 %: keep the biomarker loss (fewer changes)
            w = _res[win]
            print(f"  WINNER: {win} ({TEST_SETTINGS[win]['label']}) -- redness {w['mean_redness'] / red8:.0%} of round 8's, "
                  f"no-change grade changed {w['no_change']['grade_changed_rate']['eyepacs']:.0%} (round 8: {flip8:.0%}).")
            if win == 'B':
                print('  The biomarker regressor was a main source of the red: B (biomarker loss off) beat A clearly.')
            elif 'B' in _res and not _fixes('B'):
                print('  Turning the biomarker loss off (B) did not help further -- the identity loss + DiffAugment did the work.')
            if 'C' in _res and _fixes('C'):
                print('  ⚠ BUT the control C (no changes) also fixes it: extra training alone helped, so the gain is not '
                      'clearly from the changes. Still use the winner (it does at least as well), and say so in the write-up.')
            elif 'C' not in _res:
                print('  (Control C not run: the gain is attributed to the changes without that check -- optional.)')
            w8 = r8['graders']['eyepacs']['mean_agreement']
            if w['graders']['eyepacs']['mean_agreement'] < w8:
                print(f"  Stage control fell ({w8:.0%} -> {w['graders']['eyepacs']['mean_agreement']:.0%}): expected if the "
                      'streaks were what moved the grade in round 8 -- fine-tuning (notebook 06) is where grade control is judged.')
            if _missing and set(_missing) & {'A', 'B'}:
                print(f"  Provisional: {', '.join(k for k in _missing if k in 'AB')} not run yet.")
            print(f"  NEXT: look at the sample grids below for streaks, then set PRETRAIN_MODE = 'full' and FULL_RUN = '{win}' "
                  'in the round-9 settings cell and Run all (fresh 80-epoch pretraining, ~19 h, resumes after a disconnect).')
            with open(os.path.join(TESTS_DIR, 'verdict.json'), 'w') as f:
                json.dump({'winner': win, 'settings': TEST_SETTINGS[win], 'redness_vs_round8': w['mean_redness'] / red8,
                           'runs': list(_res)}, f, indent=2)
        # Last sample grid of each run (rows: held-out images; columns: input, then stages 0-4)
        for k, d in [('round 8', R8_NONLONG_DIR)] + [(k, test_dir(k)) for k in TEST_SETTINGS if k in _res]:
            _s = sorted(glob.glob(f'{d}/samples/epoch_*.jpg'), key=lambda p: int(p.split('_')[-1].split('.')[0]))
            if _s:
                print(f'\n{k}: {os.path.basename(_s[-1])}')
                display(IPImage(_s[-1], width=900))


## Next

Open **notebook 06** (same `MODULE1_SOURCE`) and run it: with `USE_NONLONG_PRETRAIN = True` (its
default) it fine-tunes from this notebook's `module2_runs/nonlong_pretrain{RUN_TAG}/final-G.ckpt` /
`final-D.ckpt` on FIRE + LongDR + Tianjin. Optionally run 06 a second time with it `False`, as the
ablation that shows whether EyePACS pretraining helps.

Caveats: biomarkers come from Module 1's *predicted* masks (not manual annotations), and
Module 1 does not segment IRMA/neovascularisation, so the stage-4 profile lacks NV.

## Before closing the runtime

Files written to Drive upload in the background. Run this last cell and wait for the ✓ before
disconnecting or deleting the runtime -- otherwise large files written at the end (model weights,
caches) can be lost.

In [ ]:
from google.colab import drive
drive.flush_and_unmount()
print('✓ Everything is uploaded to Google Drive -- safe to disconnect the runtime.')